# Round 4 Agent1 Specific Alpha

Purpose: mine data-only patterns first, then named-bot behavior, then map only the robust signals into `ROUND_4/agent1` candidate traders.

Working assumptions used here:
- `state.own_trades` and `state.market_trades` are disjoint in the local backtester.
- HYDRO can use the union for diagnostics/signals because our own fills still expose the named counterparty.
- VELVET stays public-market only because including our own fills worsened the existing HYPER/BOTVELVET split.


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('../../data/ROUND_4')
prices = []
trades = []
for day in (1, 2, 3):
    p = pd.read_csv(DATA / f'prices_round_4_day_{day}.csv', sep=';')
    prices.append(p)
    t = pd.read_csv(DATA / f'trades_round_4_day_{day}.csv', sep=';')
    t['day'] = day
    trades.append(t)
prices = pd.concat(prices, ignore_index=True)
trades = pd.concat(trades, ignore_index=True).rename(columns={'symbol': 'product'})
prices['spread'] = prices['ask_price_1'] - prices['bid_price_1']
mid_index = {
    (day, product): g.sort_values('timestamp').set_index('timestamp')['mid_price']
    for (day, product), g in prices.groupby(['day', 'product'])
}

def mid_at(day, product, timestamp):
    s = mid_index.get((day, product))
    if s is None:
        return None
    loc = s.index.searchsorted(timestamp)
    if loc >= len(s) or int(s.index[loc]) != int(timestamp):
        return None
    return float(s.iloc[loc])


## Data-only alpha

Strongest clean data-only pattern found: VELVET tight spread is bullish. Across all public days, spread <= 1 had n=186, +1.874 mid-to-mid at 100ms, 91.4% hit; spread <= 2 had n=1218, +1.248 at 100ms, 81.5% hit. Standalone crossing still loses too much to exit spread, so the implementation treats it only as a small threshold/tilt candidate, not as a primary strategy.

Secondary data-only pattern: VEV_5200 spread <= 1 was +0.761 at 100ms, but the event count is much lower and the edge did not survive as a high-value trader overlay in candidate testing.


In [ ]:
rows = []
for product in ['VELVETFRUIT_EXTRACT', 'HYDROGEL_PACK', 'VEV_5200', 'VEV_5300']:
    pp = prices[prices['product'] == product].copy()
    for h in [100, 300, 1000]:
        pp[f'f{h}'] = pp.groupby('day')['mid_price'].shift(-(h // 100))
        pp[f'd{h}'] = pp[f'f{h}'] - pp['mid_price']
    for spread in [1, 2, 3, 4, 5, 6, 8, 10, 16]:
        sub = pp[pp.spread <= spread]
        if len(sub) < 30:
            continue
        rows.append({
            'product': product,
            'pattern': f'spread<={spread}',
            'n': len(sub),
            'm100': sub['d100'].mean(),
            'm300': sub['d300'].mean(),
            'm1000': sub['d1000'].mean(),
            'hit100': (sub['d100'] > 0).mean(),
            'days': sub['day'].nunique(),
        })
spread_alpha = pd.DataFrame(rows)
spread_alpha.sort_values(['product', 'm100'], ascending=[True, False]).groupby('product').head(4)


## Bot-side and bot-pair alpha

Important nuance: a bot appearing as the seller is not necessarily bearish. VELVET Mark 49 selling to Mark 67 is bullish for the next tick; this is why the existing signed bot-score treats `seller alpha` with the opposite sign. The strongest durable pairs were:

- `VELVET Mark 67 -> Mark 49/22`: raw +1.98 at 100ms, all 3 days. Already captured by BOTVELVET.
- `HYDRO Mark 14 -> Mark 38`: weaker raw per-print but highly useful as a position trigger once unioned with own fills. Captured by HYPER-style HYDRO12.
- `HYDRO Mark 38 <-> Mark 22`: rare, only about 19 public events with abs raw around 3.1; tested as a pair boost but kept out unless rank improves.
- Wing-to-5200: Mark 22/Mark 01 wing clusters are sparse in public data, but agent2's direct rank showed a real +165 on VEV_5200, so agent1 tests it as an optional overlay.


In [ ]:
parts = []
for side, col, sign in [('buy', 'buyer', 1), ('sell', 'seller', -1)]:
    x = trades.copy()
    x['bot'] = x[col]
    x = x[x.bot != 'SUBMISSION']
    x['cur'] = [mid_at(r.day, r.product, int(r.timestamp)) for r in x.itertuples(index=False)]
    for h in [100, 300, 1000]:
        x[f'f{h}'] = [mid_at(r.day, r.product, int(r.timestamp) + h) for r in x.itertuples(index=False)]
        x[f'signed{h}'] = sign * (x[f'f{h}'] - x['cur'])
    parts.append(x[['day', 'timestamp', 'product', 'bot', 'quantity', 'signed100', 'signed300', 'signed1000']].assign(side=side))
bot_side = pd.concat(parts).dropna()
bot_side_summary = bot_side.groupby(['product', 'bot', 'side']).agg(
    n=('signed100', 'size'),
    q=('quantity', 'sum'),
    m100=('signed100', 'mean'),
    m300=('signed300', 'mean'),
    m1000=('signed1000', 'mean'),
    hit100=('signed100', lambda s: (s > 0).mean()),
    days=('day', 'nunique'),
).reset_index()
bot_side_summary[bot_side_summary.n >= 15].sort_values('m100', ascending=False).head(20)


In [ ]:
pair = trades.copy()
pair['cur'] = [mid_at(r.day, r.product, int(r.timestamp)) for r in pair.itertuples(index=False)]
pair['f100'] = [mid_at(r.day, r.product, int(r.timestamp) + 100) for r in pair.itertuples(index=False)]
pair = pair.dropna()
pair['raw100'] = pair['f100'] - pair['cur']
pair_summary = pair.groupby(['product', 'buyer', 'seller']).agg(
    n=('raw100', 'size'),
    q=('quantity', 'sum'),
    raw100=('raw100', 'mean'),
    days=('day', 'nunique'),
).reset_index()
pair_summary[pair_summary.n >= 8].assign(absmean=lambda d: d.raw100.abs()).sort_values('absmean', ascending=False).head(20)


## Implementation mapping

Candidate ladder in `ROUND_4/agent1`:

- `trader_AGENT1_HYDRO12.py`: BOTVELVET VELVET + HYPER-style HYDRO union/12-tick tilt.
- `trader_AGENT1_HYDRO14.py`: same with 14-tick HYDRO tilt.
- `trader_AGENT1_HYDRO12_TIGHT.py`: adds small VELVET tight-spread tilt.
- `trader_AGENT1_HYDRO12_WING5200.py`: adds 5200 wing overlay.
- `trader_AGENT1_COMBO.py`: tight + wing + HYDRO12.
- `trader_AGENT1_HYDRO12_PAIR.py`: rare HYDRO Mark38/Mark22 pair boost.

Promotion rule: top-level best must inline the chosen implementation and must not import `ROUND_4.agent1`.


In [ ]:
# Ranking commands used during this iteration:
# uv run rank --show-per-product --trader agent1/trader_AGENT1_HYDRO12.py
# uv run rank --show-per-product --trader agent1/trader_AGENT1_HYDRO12_WING5200.py
# uv run rank --show-per-product --trader trader_AGENT1_BEST.py
# uv run rank --show-per-product
# uv run check_overfit --all --by-pnl
